# 2. Pré-processamento dos Dados — Thermal Predictor

Agora que a planilha já foi reestruturada e os dados já foram explorados, é possível iniciar o pré-processamento dos dados. Nessa seção haverá a o tratamento de missing values, identificação de outliers e a correção de colunas e linhas com erro.

### Para que serve o Pré-processamento dos Dados?

O pré-processamento dos dados serve para organizar, limpar e transformar os dados brutos, corrigindo inconsistências, tratando valores ausentes e removendo informações inadequadas. Ele impacta diretamente a fase de preparação dos dados, pois garante que as informações estejam padronizadas e em um formato adequado para a análise e aplicação dos modelos, contribuindo para resultados mais confiáveis. 

### Alterações diretas na planilha

Antes de iniciar todo o processo de limpeza dos dados, a planilha dos dados foi alterada diretamente no excel, devido a erros de digitação e a mal formatação da planilha.
Principais alterações:
- Correção do número do componente, em que havia um erro no comando no excel, na qual a ordem dos números contia um erro, resultando em #REF!;

- Conversão do 'Monitor', presente na coluna de 'active_limit' para 1000. Essa alteração teve como finalidade manter um padrão para apresentar apenas números nessa coluna, evitando o uso de strings. O número 1000 foi escolhido, pois como o 'Monitor' significa que não há um limite de temperatura, foi escolhido um número que seria tecnicamente impossível de ser alcançada nesses testes;

- Na planilha havia número em formato de porcentagem, o que prejudicaria a compreensão da tabela, portanto foi alterada para o formato númerico padrão;

- As linhas 92 a 93 das Runs 1 até 18 foram coladas no fim da planilha, caso não retiradas, poderiam comprometer com a visualização do DataFrame.

## 1. Carregar planilha já reestruturada

In [ ]:
import pandas as pd
df = pd.read_csv('../assets/dataset_long.csv')

# 2. Tratamento de dados nulos

Na reestruturação da planilha, os valores nulos de temperatura já foram retirados. Essa foi considerada pelo grupo a melhor estratégia, pois o tipo de teste pode alterar significativamente as temperaturas do fogão. Dessa forma, inserir valores como média, mediana ou moda poderia impactar significativamente a análise dos dados, principalmente na fase de preparação dos dados.

Entretanto, ainda existem alguns dados nulos que precisam ser tratados, como os valores referentes ao tipo de teste. Nesses casos, será inserido o valor **"Não informado"**, mantendo os registros na base sem atribuir uma informação que não foi registrada.

In [10]:
# Definir as colunas categóricas que serão alteradas

colunas_categoricas = [
    'reason',
    'blower_direction',
    'housing',
    'board_position',
    'run_case',
    'losses',
    'deflector_hole',
    'material_coil_support'
]

# Preencher valores nulos nas colunas categóricas com 'Não informado'

df[colunas_categoricas] = df[colunas_categoricas].fillna('Não informado')

# Detectar linhas com valores 'Não informado' nas colunas categóricas
linhas_com_nao_informado = df[
    df[colunas_categoricas].eq('Não informado').any(axis=1)
]
# Exibir o número de linhas com valores 'Não informado' e as próprias linhas
print(f"Linhas com 'Não informado': {len(linhas_com_nao_informado)}")
linhas_com_nao_informado

Linhas com 'Não informado': 552


,run_id,tc_number,tc_name,key_area,active_limit,temp,reason,blower_direction,housing,board_position,run_case,losses,deflector_hole,material_coil_support
0,Run 1,1,"Door Lock Motor Ambient 1/2"" Below",Mainback,155,99.35,Não informado,To back,Open,Center,Pyro,No Losses,Open,Aluminium
1,Run 1,2,Door Lock Microswitch Body,Mainback,136,114.72,Não informado,To back,Open,Center,Pyro,No Losses,Open,Aluminium
2,Run 1,3,Convect Motor Winding,Mainback,165,89.93,Não informado,To back,Open,Center,Pyro,No Losses,Open,Aluminium
3,Run 1,7,Storage Drawer (Metal Above),Mainback,150,167.87,Não informado,To back,Open,Center,Pyro,No Losses,Open,Aluminium
4,Run 1,8,"L2, Wire, Power Cord",Mainback,60,53.65,Não informado,To back,Open,Center,Pyro,No Losses,Open,Aluminium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1690,Run 23,74,Left Encoder,CT Knobs,105,43.19,New Baseline - Hole Tray,To Front,Open - Updated,Center,Broil C2,Não informado,Closed,Aluminium
1691,Run 23,75,Left Center Encoder,CT Knobs,105,57.36,New Baseline - Hole Tray,To Front,Open - Updated,Center,Broil C2,Não informado,Closed,Aluminium
1692,Run 23,76,Ambient Between Right Encoders,CT Knobs,105,47.65,New Baseline - Hole Tray,To Front,Open - Updated,Center,Broil C2,Não informado,Closed,Aluminium
1693,Run 23,77,Right Center Encoder,CT Knobs,105,40.66,New Baseline - Hole Tray,To Front,Open - Updated,Center,Broil C2,Não informado,Closed,Aluminium


## 3. Tratamento de Outliers

Não é possível remover os outliers deste conjunto de dados, pois essa ação poderia comprometer a aplicação do machine learning. Cada tipo de teste apresenta comportamentos diferentes para cada peça e, por isso, é esperado que existam padrões distintos entre os testes. Essa variação natural resulta em valores considerados outliers quando os dados são analisados em conjunto, mas que representam corretamente as condições específicas de cada teste.

Um exemplo são os testes com `run_case` **Pyro**, que apresentam temperaturas muito baixas nas Coil temperatures, próximas de 50 °C. Em contraste, os testes com `run_case` **Broil C2** apresentam temperaturas muito altas, que podem chegar a aproximadamente 220 °C. Essas diferenças não indicam necessariamente erros ou anomalias nos dados, mas sim comportamentos próprios de cada tipo de teste.

Portanto, os outliers não devem ser removidos, pois são informações relevantes para que o modelo de machine learning aprenda as diferentes condições e comportamentos observados nos testes.

In [24]:
# Calcular o primeiro e terceiro quartil (Q1 e Q3) da temperatura para cada tipo de termopar
q1_por_tc = df.groupby('tc_name')['temp'].transform('quantile', 0.25)
q3_por_tc = df.groupby('tc_name')['temp'].transform('quantile', 0.75)

# Calcular o intervalo interquartil (IQR)
iqr_por_tc = q3_por_tc - q1_por_tc

# Definir o limite inferior e superior para identificar possíveis outliers
limite_inferior = q1_por_tc - 1.5 * iqr_por_tc
limite_superior = q3_por_tc + 1.5 * iqr_por_tc

# Indentidica como outlier os valores abaixo do limite inferior ou acima do limite superior
outliers = (df['temp'] < limite_inferior) | (df['temp'] > limite_superior)

# Filtrar os outliers, agrupá-los por tc_name e contar o total em cada grupo
total_outliers_por_tc = (
    df.loc[outliers]
    .groupby('tc_name')
    .size()
    .reset_index(name='total_outliers')
    .sort_values('total_outliers', ascending=False)
)

# Exibir o total de outliers por tipo de termopar
total_outliers_por_tc

,tc_name,total_outliers
47,IPC Vertical Duct Inlet (Bottom),15
5,AC Blower Housing,13
19,Coil FR,12
51,Power On Indicator Body,11
62,Top Duct Inlet,10
...,...,...
20,Convect Motor Winding,1
46,IPC Housing Surface,1
42,IPC Blower (Air Outlet Temp),1
38,IGBT 6,1


## 4. Tratamento de dados confusos ou insuficientes

Durante a análise, foram identificados dados que não puderam ser interpretados com segurança, pois a equipe não possui informações suficientes para confirmar seu significado. Nesses casos, é importante registrar a inconsistência e evitar a criação de valores ou interpretações que possam comprometer a análise. 

### 4.1 Active Limit do Turbo Duct / AC Blower Outlet

O `active_limit` desse componente está registrado como **"Monitor (150 °C)"**. Essa informação é confusa, pois não está claro se o componente deve apenas ser monitorado, sem possuir uma temperatura limite definida, ou se o seu limite de temperatura é de 150 °C. Como não há informações suficientes para decidir entre essas interpretações, o dado deve ser mantido conforme registrado na planilha original. 
O valor irá ser trocado por 150, por enquanto, a fim de manter o padrão das temperaturas em números, porém poderá ser trocado futuramente.

### 4.2 Componentes sem informações

Também foram encontrados componentes que ocupam uma linha inteira em branco ou apresentam pouquíssimas informações. Um exemplo é o **Dedicated IPC/CKTP Outlet**, que está completamente em branco. Como esse componente não possui dados suficientes para contribuir com a análise, ele foi retirado por completo durante a reestruturação dos dados.

### 4.3 Oven RTD Connector Body

O componente **Oven RTD Connector Body** possui um `active_limit` de 150 °C, porém todas as temperaturas registradas ultrapassam esse limite e ficam em torno de 200 °C. Portanto, os dados indicam uma possível inconsistência entre o limite informado e as temperaturas observadas.

Além disso, existe uma observação associada ao componente:

> **ID#AAACE8Xo3xk (2026-08-07 14:52:06)**  
> *CAD for RTD assembly (W11633068) passes through the backframe; likely will not see 150 C temps so remove for early stage simulation.*

Essa observação sugere que o componente provavelmente não atingirá 150 °C e recomenda sua remoção em uma simulação inicial. No entanto, a orientação ainda é subjetiva para o tratamento dos dados deste projeto, pois não está claro se o componente deve ser removido da análise, se o `active_limit` deve ser alterado ou se os valores devem ser mantidos para representar as medições reais.

Diante dessa incerteza, o grupo optou por não remover nem alterar esses dados neste momento. O componente e sua inconsistência devem ser mantidos registrados e encaminhados para validação com a equipe responsável antes de qualquer decisão definitiva.

In [25]:
# Replace direto dos valores 'Monitor( 150 C)' por 150 na coluna 'active_limit'

df['active_limit'] = df['active_limit'].replace('Monitor( 150 C)', 150)

# Filtrar as linhas do componente Turbo Duct / AC Blower Outlet
turbo_duct_ac_blower_outlet = df[
    df['tc_name'].astype('string').str.contains(
        'Turbo Duct / AC Blower Outlet',
        case=False,
        na=False,
    )
]

# Exibir somente as 5 primeiras e as 5 últimas linhas encontradas
amostra_turbo_duct_ac_blower_outlet = pd.concat([
    turbo_duct_ac_blower_outlet.head(3),
    turbo_duct_ac_blower_outlet.tail(3),
]).drop_duplicates()

# Exibir o total de linhas encontradas
print(f"Total de linhas encontradas: {len(turbo_duct_ac_blower_outlet)}")
amostra_turbo_duct_ac_blower_outlet

Total de linhas encontradas: 56


,run_id,tc_number,tc_name,key_area,active_limit,temp,reason,blower_direction,housing,board_position,run_case,losses,deflector_hole,material_coil_support
18,Run 1,26,Turbo Duct / AC Blower Outlet,Main Vent Sys,150,86.63,Não informado,To back,Open,Center,Pyro,No Losses,Open,Aluminium
85,Run 2 - Pyro Flipped,26,Turbo Duct / AC Blower Outlet,Main Vent Sys,150,90.33,Não informado,To Front,Open,Center,Pyro,No Losses,Open,Aluminium
155,Run 3 - Pyro Fans in Low,26,Turbo Duct / AC Blower Outlet,Main Vent Sys,150,78.55,Fan velocity 1500 RPM,To back,Open,Center,Pyro,No Losses,Open,Aluminium
3866,Run 73,26,Turbo Duct / AC Blower Outlet,Main Vent Sys,150,66.17,Broil Qualitative - 16 CFM (More flow on blower),To Front,Closed,Center,Broil Qual,Miro,Closed,Plastic
3940,Run 74,26,Turbo Duct / AC Blower Outlet,Main Vent Sys,150,69.22,Broil 16 CFM,To Front,Closed,Center,Broil C2,Simone,Closed,Plastic
4014,Run 75,26,Turbo Duct / AC Blower Outlet,Main Vent Sys,150,66.83,Broil Qualitative (16 cfm) CMC Deflector,To Front,Closed,Center,Broil Qual,Miro,Closed,Plastic


## 5. Salvar Alterações

In [ ]:
from pathlib import Path

# Definir o caminho absoluto do arquivo de saída
caminho_saida = Path.cwd().parent / 'assets' / 'dataset_long.csv'

try:
    # Tentar salvar as alterações no arquivo original
    df.to_csv(caminho_saida, index=False)
    print(f"Arquivo salvo em: {caminho_saida}")
except PermissionError:
    # Salvar uma cópia caso o arquivo original esteja aberto ou protegido
    caminho_alternativo = caminho_saida.with_name('dataset_long_atualizado.csv')
    df.to_csv(caminho_alternativo, index=False)
    print("Não foi possível sobrescrever dataset_long.csv.")
    print("Feche o arquivo original ou remova a proteção contra gravação para atualizá-lo.")
    print(f"Cópia salva em: {caminho_alternativo}")

# Exibir as dimensões do DataFrame salvo
print(f"Dimensões do DataFrame: {df.shape}")

PermissionError: [Errno 13] Permission denied: '../assets/dataset_long.csv'